## Iris Data in Nueral network

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder

In [2]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [3]:
X = df.drop('target',axis=1)
y = df['target']

In [4]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
## Converting into Tensor

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.long)


In [7]:
class IrisClassifier(nn.Module):
    def __init__(self,input_dim, hidden_dim, output_dim):
        super(IrisClassifier,self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,output_dim),
            
        )
    def forward(self,X):
        return self.network(X)

In [8]:
input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = 3

In [9]:
model = IrisClassifier(input_dim,hidden_dim,output_dim)


In [10]:
criterion = nn.CrossEntropyLoss()
optimize =  optim.Adam(model.parameters(),lr=0.01)

In [12]:
# Train The model 

epoch = 500

for e in range(epoch):
    model.train()
    optimize.zero_grad()
    
    prediction = model(X_train)
    loss = criterion(prediction, y_train)
    
    loss.backward()
    optimize.step()
    
    if (e+1) % 10 == 0:
        print(f"Epoch {e+1}/{e},loss: {loss.item():.4f}")

Epoch 10/9,loss: 0.0011
Epoch 20/19,loss: 0.0010
Epoch 30/29,loss: 0.0010
Epoch 40/39,loss: 0.0009
Epoch 50/49,loss: 0.0009
Epoch 60/59,loss: 0.0008
Epoch 70/69,loss: 0.0008
Epoch 80/79,loss: 0.0007
Epoch 90/89,loss: 0.0007
Epoch 100/99,loss: 0.0007
Epoch 110/109,loss: 0.0006
Epoch 120/119,loss: 0.0006
Epoch 130/129,loss: 0.0006
Epoch 140/139,loss: 0.0006
Epoch 150/149,loss: 0.0005
Epoch 160/159,loss: 0.0005
Epoch 170/169,loss: 0.0005
Epoch 180/179,loss: 0.0005
Epoch 190/189,loss: 0.0004
Epoch 200/199,loss: 0.0004
Epoch 210/209,loss: 0.0004
Epoch 220/219,loss: 0.0004
Epoch 230/229,loss: 0.0004
Epoch 240/239,loss: 0.0004
Epoch 250/249,loss: 0.0004
Epoch 260/259,loss: 0.0003
Epoch 270/269,loss: 0.0003
Epoch 280/279,loss: 0.0003
Epoch 290/289,loss: 0.0003
Epoch 300/299,loss: 0.0003
Epoch 310/309,loss: 0.0003
Epoch 320/319,loss: 0.0003
Epoch 330/329,loss: 0.0003
Epoch 340/339,loss: 0.0003
Epoch 350/349,loss: 0.0003
Epoch 360/359,loss: 0.0002
Epoch 370/369,loss: 0.0002
Epoch 380/379,loss: 0

In [14]:
model.eval()  # sets the model to evaluation mode

with torch.no_grad():  # disables gradient calculation
    y_pred = model(X_test)
    y_pred_labels = torch.argmax(y_pred,dim =1)
    
    
    accuracy = (y_pred_labels == y_test).sum().item()/ y_test.size(0)
    print(f"Test Accuracy : {accuracy:.4f}")

Test Accuracy : 1.0000


In [20]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):

    input_data = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

    input_data = scaler.transform(input_data)

    input_data = torch.tensor(input_data, dtype=torch.float32)

    model.eval()

    with torch.no_grad():

        prediction = model(input_data)

        predicted_label = torch.argmax(prediction, dim=1).item()

    return iris.target_names[predicted_label]

In [28]:
species=  predict_iris(10.1, 5, 1,2)

print(f"Predicted species:{species}")


Predicted species:virginica


c:\Users\chilesh\miniconda3\envs\yoloenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [29]:
torch.save(model.state_dict(),"IrisClassifier.pth")